In [1]:
import os
from os.path import join
import pandas as pd
import numpy as np
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

def set_col2cat(ad, cols=[]):
    for col in cols:
        ad.obs[col] = ad.obs[col].astype('category')
        
def get_umap(ad, use_reps=[]):
    for use_rep in use_reps:
        umap_add_key = f'{use_rep}_umap'
        sc.pp.neighbors(ad, use_rep=use_rep, n_neighbors=15)
        sc.tl.umap(ad)
        ad.obsm[umap_add_key] = ad.obsm['X_umap']
    return ad

def wrap_warn_plot(adata, basis, color, **kwargs):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)
        sc.pl.embedding(adata, basis=basis, color=color, **kwargs)

def wrap_warn_comp_plot(ads, basis, colors, **kwargs):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for i, color in enumerate(colors):
            _, axes = plt.subplots(1, len(ads), figsize=(10*len(ads), 5))
            for j, ad in enumerate(ads):
                sc.pl.embedding(ad, basis=basis, color=color, ax=axes[j], show=False, **kwargs)
            plt.show()

def split_ob(ads, ad_ref, ob='obs', key='emb2'):
    len_ads = [_.n_obs for _ in ads]
    if ob=='obsm':
        split_obsms = np.split(ad_ref.obsm[key], np.cumsum(len_ads[:-1]))
        for ad, v in zip(ads, split_obsms):
            ad.obsm[key] = v
    else:
        split_obs = np.split(ad_ref.obs[key].to_list(), np.cumsum(len_ads[:-1]))
        for ad, v in zip(ads, split_obs):
            ad.obs[key] = v



In [ ]:
root_data_dir = '../../../data/processed/Simulations/'

for zi, zero_p in enumerate(['0.1', '0.2', '0.3', '0.4', '0.5'], 1):
    data_dirs = os.listdir(root_data_dir)
    foldn = list(filter(lambda x: x.startswith(f'rna-{zero_p}'), data_dirs))[0]
    data_dir = root_data_dir + foldn

    for seed in [1,2,3]:
        input_train_mod1 = sc.read_h5ad(join(data_dir, f'{seed}/ad_rna1.h5ad'))
        input_train_mod2 = sc.read_h5ad(join(data_dir, f'{seed}/ad_adt1.h5ad'))
        input_test_mod1 = sc.read_h5ad(join(data_dir, f'{seed}/ad_rna2.h5ad'))
        input_test_mod2 = sc.read_h5ad(join(data_dir, f'{seed}/ad_adt2.h5ad'))

        input_test_mod1.obs_names = [f"test-mod1-{i}" for i in range(input_test_mod1.shape[0])]
        input_test_mod2.obs_names = [f"test-mod2-{i}" for i in range(input_test_mod2.shape[0])]
        
        input_train_mod1.obs['batch'] = 's1'
        input_train_mod2.obs['batch'] = 's1'
        input_test_mod1.obs['batch'] = 's2'
        input_test_mod2.obs['batch'] = 's3'

        tmp_out_dir = f'./data/processed/simulationBE-p{zi}{seed}'  
        feat_dir = join(tmp_out_dir, 'feat')                                                      
        os.makedirs(feat_dir, exist_ok=True)

        df_feat_dims = pd.DataFrame(np.array([input_train_mod1.n_vars, input_train_mod2.n_vars]).reshape(1, -1), columns=['rna', 'adt'])
        df_feat_rna_names = pd.DataFrame(input_train_mod1.var_names, columns=['x'])
        df_feat_adt_names = pd.DataFrame(input_train_mod2.var_names, columns=['x'])
        df_feat_dims.to_csv(join(feat_dir, 'feat_dims.csv'))
        df_feat_rna_names.to_csv(join(feat_dir, 'feat_names_rna.csv'))
        df_feat_adt_names.to_csv(join(feat_dir, 'feat_names_adt.csv'))

        # each subset
        subsets = [[input_train_mod1, input_train_mod2], [input_test_mod1], [input_test_mod2]]
        mods = [['rna', 'adt'], ['rna'], ['adt']]
        
        for si in range(3):
            for fname in ['mask', 'mat', 'vec']:
                os.makedirs(join(tmp_out_dir, f'subset_{si}/{fname}'), exist_ok=True)
            tmp_dir = join(tmp_out_dir, f'subset_{si}')
            for ad,mi in zip(subsets[si], mods[si]):
                mat = ad.layers['counts']
                df_mat = pd.DataFrame(mat, index=ad.obs_names, columns=ad.var_names)
                df_mat.to_csv(join(tmp_dir, f'mat/{mi}.csv'))

                os.makedirs(join(tmp_dir, f'vec/{mi}'), exist_ok=True)
                for idx, mati in enumerate(mat):
                    pd.DataFrame(mati.reshape(1, -1)).to_csv(join(tmp_dir, 'vec/{}/{:05d}.csv'.format(mi, idx)), header=None, index=None)
                pd.DataFrame(ad.obs_names, columns=['x']).to_csv(join(tmp_dir, 'cell_names.csv'))

                # save mask
                pd.DataFrame(np.ones(ad.n_vars, dtype='int')).to_csv(join(tmp_dir, f'mask/{mi}.csv'))

In [9]:
# training command
for zi, zero_p in enumerate(['0.1', '0.2', '0.3', '0.4', '0.5'], 1):
    for seed in [1,2,3]:
        for ecnt, ep in enumerate([60, 800]):
            training_command = f'CUDA_VISIBLE_DEVICES=2 python run.py --exp e{ecnt} --task simulationBE-p{zi}{seed} --epoch_num {ep}'
            print(training_command)

for zi, zero_p in enumerate(['0.1', '0.2', '0.3', '0.4', '0.5'], 1):
    for seed in [1,2,3]:
        for ecnt, ep in enumerate([60, 800]):
            if ep == 800:
                for ep2 in [100, 200, 400, 800]:
                    run_command = 'CUDA_VISIBLE_DEVICES=2 python run.py --task simulationBE-p{}{} --act predict_all_latent_bc --init_model sp_{:08d} --exp e{}'.format(zi, seed, ep2-1, ecnt)
                    print(run_command)
            else:
                run_command = 'CUDA_VISIBLE_DEVICES=2 python run.py --task simulationBE-p{}{} --act predict_all_latent_bc --init_model sp_{:08d} --exp e{}'.format(zi, seed, ep-1, ecnt)
                print(run_command)

In [10]:
def load_zu(_dir):
    zs = []

    for fi in sorted(os.listdir(_dir)):
        dfi = pd.read_csv(join(_dir, fi), header=None)
        zs.append(dfi.values)
    zs = np.vstack(zs)
    z, u = zs[:, :-2], zs[:, -2:]
    return z, u

def wrap_warn_plot(adata, basis, color, **kwargs):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)
        sc.pl.embedding(adata, basis=basis, color=color, **kwargs)

def get_umap(ad, use_reps=[]):
    for use_rep in use_reps:
        umap_add_key = f'{use_rep}_umap'
        sc.pp.neighbors(ad, use_rep=use_rep, n_neighbors=15)
        sc.tl.umap(ad)
        ad.obsm[umap_add_key] = ad.obsm['X_umap']
    return ad

In [5]:
from batchCorr import HARMONY

In [11]:
root_data_dir = '../../../data/processed/Simulations/'
result_dir = './result/'

In [12]:
ep = 60 # also tested other epochs, 60-epoch work well
model_pt = 'sp_{:08d}'.format(ep-1)
e = int(ep >= 100)

for zi, zero_p in enumerate(['0.1', '0.2', '0.3', '0.4', '0.5'], 1):   # ['0.1', '0.2', '0.3', '0.4', '0.5']
    data_dirs = os.listdir(root_data_dir) 
    foldn = list(filter(lambda x: x.startswith(f'rna-{zero_p}'), data_dirs))[0]
    data_dir = root_data_dir + foldn

    for seed in [1, 2, 3]:
        input_train_mod1 = sc.read_h5ad(join(data_dir, f'{seed}/ad_rna1.h5ad'))
        input_train_mod2 = sc.read_h5ad(join(data_dir, f'{seed}/ad_adt1.h5ad'))
        input_test_mod1 = sc.read_h5ad(join(data_dir, f'{seed}/ad_rna2.h5ad'))
        input_test_mod2 = sc.read_h5ad(join(data_dir, f'{seed}/ad_adt2.h5ad'))

        input_test_mod1.obs_names = [f"test-mod1-{i}" for i in range(input_test_mod1.shape[0])]
        input_test_mod2.obs_names = [f"test-mod2-{i}" for i in range(input_test_mod2.shape[0])]

        input_train_mod1.obs['batch'] = 's1'
        input_train_mod2.obs['batch'] = 's1'
        input_test_mod1.obs['batch'] = 's2'
        input_test_mod2.obs['batch'] = 's3'

        tmp_dir = f'{result_dir}/simulationBE-p{zi}{seed}/e{e}/default/predict/{model_pt}'
        target_ads = [input_train_mod1, input_test_mod1, input_test_mod2]
        for i in range(3):
            z, u = load_zu(join(tmp_dir, f'subset_{i}/z/joint'))
            target_ads[i].obsm['z'] = z
            target_ads[i].obsm['u'] = u

        ad_mosaic = sc.concat(target_ads)
        # ad_mosaic = get_umap(ad_mosaic, ['z'])
        df_emb = pd.DataFrame(ad_mosaic.obsm['z'], 
                index=input_train_mod1.obs_names.to_list() + input_test_mod1.obs_names.to_list() + input_test_mod2.obs_names.to_list())
        
        os.makedirs(f'../../../results/embeddings/MIDAS/Simulations/{foldn}/{seed}', exist_ok=True)
        df_emb.to_csv(f'../../../results/embeddings/MIDAS/Simulations/{foldn}/{seed}/df_emb.csv')


Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
	Completed 5 / 10 iteration(s).
	Completed 6 / 10 iteration(s).
	Completed 7 / 10 iteration(s).
	Completed 8 / 10 iteration(s).
	Completed 9 / 10 iteration(s).
	Completed 10 / 10 iteration(s).
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
Reach convergence after 3 iteration(s).
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
	Completed 5 / 10 iteration(s).
	Completed 6 / 10 iteration(s).
	Completed 7 / 10 iteration(s).
	Completed 8 / 10 iteration(s).
	Completed 9 / 10 iteration(s).
	Completed 10 / 10 iteration(s).
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 /